In [1]:
# FRA red-team v5 / ICON setup — files uploaded through the Files pane
import hashlib, os, pathlib, subprocess, sys, zipfile
ROOT = pathlib.Path('/content/fra-v5-icon')
OUT = pathlib.Path('/content/fra-v5-icon-results')
ROOT.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
ARCHIVE_SHA = 'ca160e76210755d6497e1120aed443664f343256b26d09ec4e50ba67eb980080'
BUNDLE_SHA = '143b84e94d07c62344137e14ba13fbedc436f15f5c8baa5c21ef0aa15142125c'
ADAPTER_SHA = '72f82ddc594ff07000f0caadcd95b645219faac0929f21d63ab08542cc9c953f'
for name, expected in [('fra-v5-icon-source.zip', ARCHIVE_SHA), ('fra-v5-icon-colab-bundle.zip', BUNDLE_SHA)]:
    archive = pathlib.Path('/content') / name
    assert archive.exists(), f'Upload not finished: {name}'
    actual = hashlib.sha256(archive.read_bytes()).hexdigest()
    assert actual == expected, (name, actual)
    with zipfile.ZipFile(archive) as z:
        z.extractall(ROOT)
    print('VERIFIED', name, actual, flush=True)
assert hashlib.sha256((ROOT / 'icon_supplement.py').read_bytes()).hexdigest() == ADAPTER_SHA
os.chdir(ROOT)
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'ml/requirements.txt'], text=True, capture_output=True)
(OUT / 'install.txt').write_text(result.stdout + result.stderr)
assert result.returncode == 0, result.stderr
result = subprocess.run([sys.executable, '-m', 'pytest', 'ml/tests', '-q'], text=True, capture_output=True)
(OUT / 'pytest.txt').write_text(result.stdout + result.stderr)
print(result.stdout, result.stderr)
assert result.returncode == 0, 'Existing tests failed'
print('SETUP_AND_EXISTING_TESTS_PASSED', flush=True)


VERIFIED fra-v5-icon-source.zip ca160e76210755d6497e1120aed443664f343256b26d09ec4e50ba67eb980080
VERIFIED fra-v5-icon-colab-bundle.zip 143b84e94d07c62344137e14ba13fbedc436f15f5c8baa5c21ef0aa15142125c
......................................................                   [100%]
54 passed in 2.45s
 
SETUP_AND_EXISTING_TESTS_PASSED


In [2]:
# Both model comparisons run only on the Google Colab CPU runtime.
import pathlib, subprocess, sys
ROOT = pathlib.Path('/content/fra-v5-icon')
OUT = pathlib.Path('/content/fra-v5-icon-results')
for lag in (6, 12):
    target = OUT / f'lag{lag}'
    target.mkdir(exist_ok=True)
    cmd = [sys.executable, '-u', str(ROOT/'experiment_v5_icon.py'), '--repo', str(ROOT), '--out', str(target), '--lag', str(lag), '--archive-sha', ARCHIVE_SHA, '--adapter-sha', ADAPTER_SHA]
    with (target/'run.log').open('w') as log:
        proc = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in proc.stdout:
            log.write(line)
            log.flush()
            print(line, end='', flush=True)
        code = proc.wait()
    print('LAG_RETURN_CODE', lag, code, flush=True)
    assert code == 0, f'lag {lag} failed; inspect run.log'
print('BOTH_LAGS_COMPLETE', flush=True)


BASE features from metadata: 45 (order-verified)
BASELINE 45 + SUPPLEMENTAL 7 = 52 at lag 6
ICON source contract OK: 16008 hourly UTC targets, kn/deg columns, icon_global
ALIGNMENT day1-vs-METAR r: shift-24h=0.360 shift0=0.782 shift+24h=0.382 n=15312
Flagged 103 recorded hours as a stuck feed
Features 500/4123 issue times
Features 1000/4123 issue times
Features 1500/4123 issue times
Features 2000/4123 issue times
Features 2500/4123 issue times
Features 3000/4123 issue times
Features 3500/4123 issue times
Features 4000/4123 issue times
Dataset 194,352 rows, 4,049 issues; 74 skipped
FULL v5 build rows: 194352 issues skipped: 74
SUPPLEMENTAL coverage over 2025+ issues: 1.0000
ISSUES candidate 2659 complete-48h 2659
JOIN OK: base block byte-identical, 7 raw supplemental cols finite with variance
ROWS train 122280 test 5136
Fitting CORE baseline (metadata base features, 2025+ full-48h issues)
Fitting CORE candidate (base+ICON-raw, identical rows)
SELECTOR blocks early 110568 blendval 5672 s

In [3]:
# Independent recomputation and historical shadow-decision evidence; no model fitting.
import hashlib, json, pathlib, shutil, subprocess, sys
import numpy as np
import pandas as pd
ROOT = pathlib.Path('/content/fra-v5-icon')
OUT = pathlib.Path('/content/fra-v5-icon-results')
all_summary = []
for lag in (6, 12):
    dest = OUT / f'lag{lag}'
    df = pd.read_parquet(dest/'per_row_traces.parquet')
    metrics = pd.read_csv(dest/'metrics.csv')
    policy = json.loads((dest/'selector_table.json').read_text())
    headline = metrics[metrics['slice'].eq('all_recorded')]
    for row in headline.itertuples():
        subset = df[df.lead_h.eq(row.lead_h)]
        values = subset[row.method].to_numpy(float)
        truth = subset.y_od.to_numpy(float)
        assert len(subset) == row.n
        assert np.isclose(((values >= .5) == truth).mean(), row.accuracy)
        assert np.isclose(((values-truth)**2).mean(), row.brier)
    # A frozen-policy transfer must be compared with a global choice on the same models.
    df['global_core'] = df['icon_blend'] if policy['global_arm'] == 'candidate' else df['baseline_blend']
    rows = []
    for lead in (1, 6, 24, 48):
        sub = df[df.lead_h.eq(lead)].copy()
        for method in ('baseline_blend','icon_blend','equalblend','selector','global_core','selector_frozen','global_arm','p_frozen_baseline','p_frozen_candidate','persistence','rules'):
            rows.append(dict(lag=lag,lead_h=lead,method=method,n=len(sub),accuracy=float(((sub[method]>=.5)==sub.y_od).mean()),brier=float(((sub[method]-sub.y_od)**2).mean())))
    summary = pd.DataFrame(rows)
    summary.to_csv(dest/'audited_headline.csv',index=False)
    all_summary.append(summary)
    # Paired whole-issue-day resampling preserves within-day rows and candidate pairing.
    cis = []
    for lead in (6,24,48):
        sub = df[df.lead_h.eq(lead)].copy()
        sub['day'] = pd.to_datetime(sub.issue_time,utc=True).dt.tz_convert('Europe/Berlin').dt.date
        for candidate,reference in [('icon_blend','baseline_blend'),('equalblend','baseline_blend'),('selector','global_core'),('selector_frozen','global_arm')]:
            sub['delta'] = (sub[candidate]-sub.y_od)**2-(sub[reference]-sub.y_od)**2
            daily = sub.groupby('day')['delta'].agg(['sum','count'])
            rng = np.random.default_rng(42)
            starts = rng.integers(0,len(daily),size=(2000,int(np.ceil(len(daily)/7))))
            draws = ((starts[:,:,None] + np.arange(7)) % len(daily)).reshape(2000,-1)[:,:len(daily)]
            samples = daily['sum'].to_numpy()[draws].sum(axis=1)/daily['count'].to_numpy()[draws].sum(axis=1)
            low,high=np.quantile(samples,[.025,.975])
            cis.append(dict(lag=lag,lead_h=lead,candidate=candidate,reference=reference,n=len(sub),n_issue_days=len(daily),brier_delta=float(sub.delta.mean()),ci_low=float(low),ci_high=float(high),bootstrap_replicates=2000,seed=42))
    pd.DataFrame(cis).to_csv(dest/'paired_7day_bootstrap.csv',index=False)
    slices=[]
    for (lead,cell),sub in df[df.lead_h.isin([6,24,48])].groupby(['lead_h','selector_cell']):
        for method in ('baseline_blend','icon_blend','selector','global_core','selector_frozen','global_arm'):
            slices.append(dict(lag=lag,lead_h=int(lead),cell=cell,n=len(sub),method=method,brier=float(((sub[method]-sub.y_od)**2).mean()),accuracy=float(((sub[method]>=.5)==sub.y_od).mean())))
    pd.DataFrame(slices).to_csv(dest/'regime_scores.csv',index=False)
    models_sha=hashlib.sha256((dest/'models.joblib').read_bytes()).hexdigest()
    script_sha=hashlib.sha256((ROOT/'experiment_v5_icon.py').read_bytes()).hexdigest()
    cases=[]
    # Deterministic first example per validation-defined cell, never chosen by correctness.
    for row in df.sort_values(['issue_time','lead_h']).groupby('selector_cell',sort=True).head(1).itertuples():
        cases.append(dict(mode='historical shadow replay; not a deployed decision API',issue_time=str(row.issue_time),target_time=str(row.time),lead_h=int(row.lead_h),task='OD direction probability',cell=row.selector_cell,selected_arm=row.selector_arm,reason=row.selector_reason,eligible=bool(row.selector_eligible),selector_frozen_probability=float(row.selector_frozen),frozen_candidates=dict(baseline=float(row.p_frozen_baseline),icon=float(row.p_frozen_candidate)),core_candidates=dict(baseline=float(row.baseline_blend),icon=float(row.icon_blend)),observed_target_for_retrospective_scoring=float(row.y_od),models_sha256=models_sha,experiment_script_sha256=script_sha,delay_counterfactual_supported=False))
    (dest/'shadow_decision_examples.json').write_text(json.dumps(cases,indent=2))
    audit=dict(lag=lag,headline_rows_recomputed=len(headline),scored_rows=len(df),global_arm_selected_on_september=policy['global_arm'],models_sha256=models_sha,experiment_script_sha256=script_sha,limitations=['October was previously used in development; exploratory only.','Circular 7-day issue-date blocks preserve pairing; October has few blocks, so intervals remain exploratory.','No test-outcome-based policy selection.','New full-trajectory OD timing scores are not computed from label-filtered traces; do not reinterpret changed-flag accuracy as event recall.','Runway-18 labels are inferred, not actual closure logs.','BTS does not support FRA flight-level delay counterfactuals.'])
    (dest/'audit.json').write_text(json.dumps(audit,indent=2))
    print('AUDIT_PASS',lag,len(headline),'headline rows',len(df),'predictions',flush=True)
    print(summary.round(5).to_string(index=False),flush=True)
    print(pd.DataFrame(cis).round(5).to_string(index=False),flush=True)
combined=pd.concat(all_summary,ignore_index=True)
combined.to_csv(OUT/'audited_headline_all_lags.csv',index=False)
subprocess.run([sys.executable,'-m','pip','freeze'],stdout=(OUT/'requirements-lock.txt').open('w'),check=True)
shutil.copy2(ROOT/'experiment_v5_icon.py',OUT/'experiment_v5_icon.py')
shutil.copy2(ROOT/'icon_supplement.py',OUT/'icon_supplement.py')
print('BOTH_LAGS_AUDITED',flush=True)

# Separate runway-18 hourly scores and complete-window sensitivity.
r18_scores=[]
r18_events=[]
for lag in (6,12):
    dest=OUT/f'lag{lag}'
    r=pd.read_parquet(dest/'r18_traces.parquet')
    for lead in (1,6,24,48):
        s=r[r.lead_h.eq(lead)]
        for method in ('p18_baseline','p18_candidate','rule_18_usable','persistence_18'):
            r18_scores.append(dict(lag=lag,lead_h=lead,method=method,n=len(s),accuracy=float(((s[method]>=.5)==s.y_r18).mean()),brier=float(((s[method]-s.y_r18)**2).mean())))
    for horizon in (1,6,24):
        eligible=[]
        for issue,g in r[r.lead_h.le(horizon)].groupby('issue_time'):
            if sorted(g.lead_h.tolist())==list(range(1,horizon+1)):
                eligible.append(g)
        for method in ('p18_baseline','p18_candidate','rule_18_usable','persistence_18'):
            tp=fp=fn=tn=0
            for g in eligible:
                actual=bool((g.y_r18<.5).any()); pred=bool((g[method]<.5).any())
                tp+=int(actual and pred); fp+=int(not actual and pred)
                fn+=int(actual and not pred); tn+=int(not actual and not pred)
            r18_events.append(dict(lag=lag,horizon_h=horizon,method=method,eligible_windows=len(eligible),total_issue_windows=107,excluded_windows=107-len(eligible),tp=tp,fp=fp,fn=fn,tn=tn,label_type='weather-rule proxy; overlapping windows'))
pd.DataFrame(r18_scores).to_csv(OUT/'r18_hourly_metrics.csv',index=False)
pd.DataFrame(r18_events).to_csv(OUT/'r18_complete_window_events.csv',index=False)
print('R18_HOURLY',pd.DataFrame(r18_scores).round(6).to_csv(index=False))
print('R18_EVENTS',pd.DataFrame(r18_events).to_csv(index=False))
# Standalone chart: matched comparisons only, separately by lag.
import matplotlib.pyplot as plt
fig,axes=plt.subplots(1,2,figsize=(11,4),sharey=True)
for ax,lag in zip(axes,(6,12)):
    for method in ('baseline_blend','icon_blend','equalblend','selector'):
        s=combined[(combined.lag==lag)&(combined.method==method)]
        ax.plot(s.lead_h,s.brier,marker='o',label=method)
    ax.set_title(f'October exploratory / lag proxy {lag} h')
    ax.set_xlabel('Forecast lead (hours)'); ax.set_xticks([1,6,24,48]); ax.grid(alpha=.2)
axes[0].set_ylabel('OD Brier score (lower is better)')
axes[1].legend(fontsize=8)
fig.suptitle('Matched v5 vs raw ICON and scenario selection — not v6 comparison')
fig.tight_layout(); fig.savefig(OUT/'colab_comparison.svg'); fig.savefig(OUT/'colab_comparison.png',dpi=160); plt.close(fig)
print('AUDITED_CSV_BEGIN')
print(combined.to_csv(index=False))
print('AUDITED_CSV_END')
print('EVIDENCE_AUDIT_COMPLETE')


AUDIT_PASS 6 40 headline rows 5068 predictions
 lag  lead_h             method   n  accuracy   brier
   6       1     baseline_blend 105   0.91429 0.06876
   6       1         icon_blend 105   0.90476 0.07567
   6       1         equalblend 105   0.90476 0.07204
   6       1           selector 105   0.90476 0.07566
   6       1        global_core 105   0.90476 0.07567
   6       1    selector_frozen 105   0.89524 0.07918
   6       1         global_arm 105   0.89524 0.07918
   6       1  p_frozen_baseline 105   0.88571 0.07966
   6       1 p_frozen_candidate 105   0.89524 0.07918
   6       1        persistence 105   0.96190 0.03810
   6       1              rules 105   0.96190 0.03810
   6       6     baseline_blend 107   0.82243 0.14923
   6       6         icon_blend 107   0.81308 0.15029
   6       6         equalblend 107   0.82243 0.14965
   6       6           selector 107   0.81308 0.15029
   6       6        global_core 107   0.81308 0.15029
   6       6    selector_frozen 107